# Supervised FineTuning (SFT) con LoRa sobre un modelo pequeño Qwen

SFT en modelos pequeños:

[Explicación de SFT con LoRA](https://kaifimtz.medium.com/fine-tuning-large-language-models-sft-lora-and-qlora-explained-d97bd4c1504e)


Instalemos las librerías necesarias a usar e instanciemos los datos desde Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
!pip install -q trl torchao

Importemos las librerías a usar

In [ ]:
import torch
VRAM_GB = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM (GB):", VRAM_GB)
print("Soporta BF16:", torch.cuda.is_bf16_supported())

In [ ]:
import json, random
from collections import Counter, defaultdict
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from datasets import Dataset
from peft import LoraConfig, get_peft_model
from safetensors.torch import load_file
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Trainer,
    TrainingArguments,
)

Definamos los parámetros del modelo

In [ ]:
MODEL_ID = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
SFT_PATH = "/content/drive/MyDrive/GenAI/sft.jsonl"
MAX_LEN = 3200
OUT_DIR = "/content/drive/MyDrive/GenAI/e2/checkpoints/sft_lora"
USE_BF16 = torch.cuda.is_bf16_supported()
SEED = 42

Importamos el tokenizer de Qwen

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
tokenizer

Carguemos los datos

In [ ]:
rows = [json.loads(line) for line in open(SFT_PATH, encoding='utf-8') if line.strip()]
rows[0], "#################################### uwu ##############################", rows[1:3]

In [ ]:
by_diff = defaultdict(list)
for tid, diff in sorted({(r["task_id"], r["difficulty"]) for r in rows}):
    by_diff[diff].append(tid)
{k: len(v) for k, v in by_diff.items()}

In [ ]:
rng = random.Random(SEED)
for tids in by_diff.values():
    rng.shuffle(tids)
{k: v[:2] for k, v in by_diff.items()}

Usemos el 10% del dataset para validation (estratificado por dificultad y agrupado por task_id). Por tanto, usaremos un 90% para train split

In [ ]:
VAL_SPLIT_PCT = 0.10
TRAIN_SPLIT_PCT = 1 - VAL_SPLIT_PCT

val_ids = set()
for diff, tids in sorted(by_diff.items()):
    n_val = max(1, int(round(len(tids) * VAL_SPLIT_PCT)))
    val_ids.update(tids[:n_val])
len(val_ids)

Obtenemos los datos crudos

In [ ]:
train_raw = [r for r in rows if r["task_id"] not in val_ids]
train_raw[0]

In [ ]:
val_raw = [r for r in rows if r["task_id"] in val_ids]
val_raw[0]

In [ ]:
print(f"Train: {len(train_raw)} trazas | Val: {len(val_raw)} trazas")

## Proporciones del dataset

In [ ]:
DIFFS = ["easy", "medium", "hard", "expert"]
tr_diff = Counter(r["difficulty"] for r in train_raw)
va_diff = Counter(r["difficulty"] for r in val_raw)

prop_df = pd.DataFrame([
    {
        "Dificultad": d,
        "Train": tr_diff[d],
        "Train (%)": round(100 * tr_diff[d] / len(train_raw), 1),
        "Val": va_diff[d],
        "Val (%)": round(100 * va_diff[d] / len(val_raw), 1),
    }
    for d in DIFFS
])
display(prop_df)

msg_counts = Counter(len(r["messages"]) for r in rows)
turn_keys = sorted(msg_counts)
est_toks = [sum(len(m["content"]) + 20 for m in r["messages"]) / 3.47 for r in rows]

fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))

x = np.arange(len(DIFFS))
axes[0].bar(x - 0.18, prop_df["Train (%)"], width=0.35, label="Train (90%)")
axes[0].bar(x + 0.18, prop_df["Val (%)"], width=0.35, label="Val (10%)")
axes[0].set_xticks(x)
axes[0].set_xticklabels(DIFFS)
axes[0].set_ylabel("%")
axes[0].set_title("Train vs Val por dificultad")
axes[0].legend()
axes[0].grid(axis="y", alpha=0.3)

axes[1].bar([f"{k} msgs" for k in turn_keys], [msg_counts[k] for k in turn_keys])
for i, k in enumerate(turn_keys):
    axes[1].text(i, msg_counts[k] + 30, f"{msg_counts[k]} ({100*msg_counts[k]/len(rows):.1f}%)", ha="center", fontsize=8)
axes[1].set_ylim(0, max(msg_counts.values()) * 1.18)
axes[1].set_title("Turnos en sft.jsonl")
axes[1].grid(axis="y", alpha=0.3)

axes[2].hist(est_toks, bins=35, edgecolor="white")
axes[2].axvline(MAX_LEN, color="red", linestyle="--", label=f"MAX_LEN={MAX_LEN}")
axes[2].set_xlabel("Tokens estimados")
axes[2].set_title("Largo de secuencia")
axes[2].legend()
axes[2].grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

## Tokenización

Podriamos usa `apply_chat_template` de HuggingFace, que viene ya con el tokenizer. Pero no quize usarlo por ahora, aunque podría ser recomendable solo por mantener la consistencia. Pero habría que verificar que realmente es correcto el método. Además, igualmente habría que enmascarar con -100 las partes que no queremos que aprenda el modelo.

In [ ]:
def tokenize_with_masking(example):
  input_ids = []
  labels_ids = []

  for msg in example["messages"]:
    role = msg["role"]
    content = msg["content"]

    # Agregamos el formato que necesita Qwen
    header_ids = tokenizer.encode(f"<|im_start|>{role}\n", add_special_tokens=False)
    body_ids   = tokenizer.encode(f"{content}<|im_end|>\n", add_special_tokens=False)

    input_ids.extend(header_ids + body_ids)

    if role == "assistant":
      # Enmascaramos encabezado "<|im_start|>assistant\n" solamente. Hay que calcular la
      # loss en la respuesta + <|im_end|>

      # Ponemos el id: -100 por cada uno de los header_ids. Luego, agregamos los
      # body_id tal cual
      labels_ids.extend([-100] * len(header_ids) + body_ids)

    else:
      # system y user (tools) roles, ponemos el id: -100 en todo, porque no
      # queremos que
      # se fije en esto. Solo que se concentr en predecir bien lo del asistente
      labels_ids.extend([-100] * (len(header_ids) + len(body_ids)))

  return {
      "input_ids": input_ids[:MAX_LEN],
      "labels": labels_ids[:MAX_LEN],
      "attention_mask": [1] * min(len(input_ids), MAX_LEN),
  }


In [ ]:
train_ds = Dataset.from_list(train_raw).map(tokenize_with_masking, remove_columns=["task_id", "difficulty", "messages"])
val_ds   = Dataset.from_list(val_raw).map(tokenize_with_masking, remove_columns=["task_id", "difficulty", "messages"])

## Verificación de tokenización correcta

In [ ]:
sample = train_ds[0]
active_ids = [tid for tid, lbl in zip(sample["input_ids"], sample["labels"]) if lbl != -100]
print("=== TOKENS CON LOSS ACTIVA (solo debe aparecer <sql> y <answer>) ===")
print(tokenizer.decode(active_ids))
print("\nTokens totales:", len(sample["input_ids"]), "| Tokens con loss:", len(active_ids))
print("Largo máximo en train_ds:", max(len(x["input_ids"]) for x in train_ds))
print("Largo máximo en val_ds:", max(len(x["input_ids"]) for x in val_ds))
assert sum(1 for x in train_ds if all(l == -100 for l in x["labels"])) == 0
assert sum(1 for x in val_ds if all(l == -100 for l in x["labels"])) == 0

## Instanciar el modelo LoRa

In [ ]:
# Cargamos modelo base en BF16 (L4/A100/G4) o FP16 (T4)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.bfloat16 if USE_BF16 else torch.float16,
    device_map="auto",
)
model.config.use_cache = False

## Inyectar adaptadores LoRA

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

Se identifican trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820

## Definición de Collator

In [ ]:
# Collator que rellena con pad_token_id en input_ids y con -100 en labels
collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    padding=True,
    label_pad_token_id=-100,
)

## Entrenar y guardar adaptador LoRA

Definimos aqui en la misma celda los hiperparámetros

In [ ]:
batch_size = 4 if VRAM_GB >= 40 else 2
grad_accum = 4 if VRAM_GB >= 40 else 8
use_ckpt = VRAM_GB < 40

training_args = TrainingArguments(
    output_dir=OUT_DIR,
    num_train_epochs=2,
    per_device_train_batch_size=batch_size,
    gradient_accumulation_steps=grad_accum,
    per_device_eval_batch_size=1,
    prediction_loss_only=True,
    eval_accumulation_steps=1,
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    warmup_steps=15,
    weight_decay=0.01,
    max_grad_norm=1.0,
    gradient_checkpointing=use_ckpt,
    gradient_checkpointing_kwargs={"use_reentrant": False} if use_ckpt else None,
    bf16=USE_BF16,
    fp16=not USE_BF16,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=40,
    save_strategy="epoch",              # Respalda en Drive al final de cada época
    save_total_limit=3,
    report_to="none",
    seed=SEED,
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collator,
)
train_result = trainer.train()

Run simple para ver si funciona el script

In [ ]:
# import gc, torch
# gc.collect()
# torch.cuda.empty_cache()

# training_args = TrainingArguments(
#     per_device_eval_batch_size=1,   # Baja los logits de 11.4 GiB -> 1.4 GiB (¡8x menos!)
#     prediction_loss_only=True,      # Solo calcula el número eval_loss; NO acumula los logits gigantes en GPU
#     output_dir=OUT_DIR,
#     max_steps=5,                      # <-- Solo 5 pasos para probar
#     per_device_train_batch_size=1,
#     gradient_accumulation_steps=4,    # <-- 4x más rápido por paso solo para testear
#     logging_steps=1,                  # <-- Imprime la loss en CADA paso (Step 1, 2, 3...)
#     eval_strategy="steps",
#     eval_steps=5,                     # <-- Prueba también que eval_loss corre sin OOM
#     save_strategy="no",               # <-- No ocupa espacio en Drive durante el test
#     gradient_checkpointing=True,
#     gradient_checkpointing_kwargs={"use_reentrant": False},
#     bf16=USE_BF16,
#     fp16=not USE_BF16,
#     learning_rate=2e-4,
#     report_to="none",
#     seed=SEED,
# )

# # Usa solo 10 ejemplos de validación para que eval_steps=5 termine en segundos:
# trainer = Trainer(
#     model=model,
#     args=training_args,
#     train_dataset=train_ds,
#     eval_dataset=val_ds.select(range(10)),
#     data_collator=collator,
# )
# trainer.train()

Guardamos el modelo en Google Drive, muy importante ! ! !

In [ ]:
## Guardar adaptador final
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
final_name = f"final_{timestamp}"
final_dir = Path(OUT_DIR) / final_name
final_dir.mkdir(parents=True, exist_ok=True)

trainer.save_model(str(final_dir))
tokenizer.save_pretrained(str(final_dir))

logs = trainer.state.log_history
with open(final_dir / "log_history.json", "w", encoding="utf-8") as f:
    json.dump(logs, f, indent=2, ensure_ascii=False)

run_meta = {
    "model_id": MODEL_ID,
    "gpu": torch.cuda.get_device_name(0),
    "vram_gb": VRAM_GB,
    "vram_peak_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2),
    "max_len": MAX_LEN,
    "seed": SEED,
    "val_split_pct": VAL_SPLIT_PCT,
    "n_train": len(train_raw),
    "n_val": len(val_raw),
    "train_loss": float(train_result.training_loss),
    "global_step": trainer.state.global_step,
    "val_ids": sorted(val_ids),
}
with open(final_dir / "run_metadata.json", "w", encoding="utf-8") as f:
    json.dump(run_meta, f, indent=2, ensure_ascii=False)

adapter_path = final_dir / "adapter_model.safetensors"
assert adapter_path.exists()
tensors = load_file(str(adapter_path))
lora_b_norms = [float(v.float().norm().item()) for k, v in tensors.items() if "lora_B" in k]
assert max(lora_b_norms) > 0
print(f"Guardado OK en {final_dir} | {adapter_path.stat().st_size / (1024**2):.2f} MB | lora_B norm mean={np.mean(lora_b_norms):.4f}")

## Evaluación

Graficamos la pérdida obtenida

In [ ]:
train_steps = [x["step"] for x in logs if "loss" in x]
train_loss  = [x["loss"] for x in logs if "loss" in x]
eval_steps  = [x["step"] for x in logs if "eval_loss" in x]
eval_loss   = [x["eval_loss"] for x in logs if "eval_loss" in x]
lr_steps    = [x["step"] for x in logs if "learning_rate" in x]
lrs         = [x["learning_rate"] for x in logs if "learning_rate" in x]
gn_steps    = [x["step"] for x in logs if "grad_norm" in x]
grad_norms  = [x["grad_norm"] for x in logs if "grad_norm" in x]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(train_steps, train_loss, label="Train Loss (Assistant-Only)")
axes[0].plot(eval_steps, eval_loss, marker="o", label=f"Val Loss ({int(VAL_SPLIT_PCT*100)}% tareas)")
axes[0].set_xlabel("Step")
axes[0].set_ylabel("Cross-Entropy Loss")
axes[0].set_title("Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(lr_steps, lrs, color="tab:green")
axes[1].set_xlabel("Step")
axes[1].set_ylabel("LR")
axes[1].set_title("Learning Rate")
axes[1].grid(True, alpha=0.3)

axes[2].plot(gn_steps, grad_norms, color="tab:orange")
axes[2].set_xlabel("Step")
axes[2].set_ylabel("Grad Norm")
axes[2].set_title("Gradient Norm")
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{OUT_DIR}/{final_name}/sft_loss_curve.png", dpi=150, bbox_inches="tight")
plt.show()